In [ ]:
import pandas as pd
from config import DEFAULT_GEF_PATH, ANALYSIS_RESULTS_DIR
from data_processing import extract_gene_expression_from_gef
from utils import ensure_dir

# ============================
# Step 1: 从 GEF 提取基因表达 (支持 Typo 搜索)
# ============================
gef_path = str(DEFAULT_GEF_PATH)
output_csv = ANALYSIS_RESULTS_DIR / "temp_snca_extraction.csv"

target_typo = "SCNA"
print("🚀 [Step 1] 正在启动 Stereopy 搜索...")
print(f"🎯 目标锁定: '{target_typo}' (怀疑是录入 Typo)")

df = extract_gene_expression_from_gef(
    gef_path=gef_path,
    bin_size=100,
    target_gene=target_typo,
)

ensure_dir(output_csv.parent)
df.to_csv(output_csv, index=False)
print(f"💾 提取成功！已保存至: {output_csv}")
print(f"   最大表达量: {df['expression'].max()}")


In [ ]:
import scanpy as sc
import pandas as pd
from config import ANALYSIS_RESULTS_DIR
from data_processing import inject_expression_by_spot_id

# ============================
# Step 2: 合并 C2L 结果并注入 SCNA
# ============================
raw_file = "bin100_raw.h5ad"
c2l_result_file = ANALYSIS_RESULTS_DIR / "results_bin100_final.h5ad"
scna_csv_path = ANALYSIS_RESULTS_DIR / "temp_snca_extraction.csv"
output_file = ANALYSIS_RESULTS_DIR / "results_bin100_complete.h5ad"

print("🚀 [Step 2] 开始构建全量基因集结果文件...")

adata_master = sc.read_h5ad(raw_file)
adata_master.var_names_make_unique()
print(f"   底座基因数: {adata_master.n_vars}")

adata_c2l = sc.read_h5ad(c2l_result_file)
scna_df = pd.read_csv(scna_csv_path)

# 迁移 Cell2location 结果
common_cells = adata_master.obs_names.intersection(adata_c2l.obs_names)
print(f"   Raw 细胞数: {adata_master.n_obs}")
print(f"   C2L 细胞数: {adata_c2l.n_obs}")
print(f"   交集细胞数: {len(common_cells)}")

if len(common_cells) < adata_c2l.n_obs:
    adata_master = adata_master[common_cells].copy()
    adata_c2l = adata_c2l[common_cells].copy()
else:
    adata_c2l = adata_c2l[adata_master.obs_names].copy()

if "q05_cell_abundance_w_sf" in adata_c2l.obsm:
    adata_master.obsm["q05_cell_abundance_w_sf"] = adata_c2l.obsm["q05_cell_abundance_w_sf"]
    print("✅ 细胞丰度迁移成功。")
else:
    print("❌ 结果文件中缺少 q05_cell_abundance_w_sf")

# 注入 SCNA
new_expr = inject_expression_by_spot_id(adata_master, scna_df, "expr_SNCA")
print(f"   SCNA 匹配成功数: {(new_expr > 0).sum()} / {adata_master.n_obs}")

# 修复 var 名字
if "SCNA" in adata_master.var_names:
    new_index = adata_master.var_names.tolist()
    idx_loc = new_index.index("SCNA")
    new_index[idx_loc] = "SNCA"
    adata_master.var_names = new_index

if "counts" not in adata_master.layers:
    adata_master.layers["counts"] = adata_master.X.copy()

adata_master.write(output_file)
print(f"🎉 已保存: {output_file}")


In [ ]:
import scanpy as sc
import pandas as pd
from config import ANALYSIS_RESULTS_DIR
from data_processing import inject_expression_by_coords

# ============================
# Step 3: 坐标修复 (KDTree + 可选旋转)
# ============================
h5ad_file = ANALYSIS_RESULTS_DIR / "results_bin100_complete.h5ad"
csv_file = ANALYSIS_RESULTS_DIR / "temp_snca_extraction.csv"
output_file = ANALYSIS_RESULTS_DIR / "results_bin100_complete_fixed.h5ad"

adata = sc.read_h5ad(h5ad_file)
scna_df = pd.read_csv(csv_file)

new_expr = inject_expression_by_coords(
    adata,
    scna_df,
    target_obs_key="expr_SNCA",
    use_rotation=True,
)

print(f"📊 修复后 obs['expr_SNCA'] 最大值: {new_expr.max()}")
if new_expr.max() > 0:
    adata.write(output_file)
    print(f"💾 修复文件已保存: {output_file}")
